# Compare with official Diffusion-TS

Run the first three code cells on a GPU runtime. This notebook reads the completed fixed/global/patch experiment from Drive, trains one official Diffusion-TS baseline, and evaluates all four generated sample sets. It saves results under the existing experiment's benchmark_diffusion_ts folder.

The comparison uses this pilot's shared data, z-score normalization, optimizer budget, validation selection, and unclipped DDIM sampler. Diffusion-TS keeps its released architecture and L1/Fourier training objective. C-FID uses one shared TS2Vec encoder. DS and PS use a PyTorch implementation of the released TimeGAN-style GRU evaluator definitions. Evaluator-repeat standard deviations describe evaluation randomness for a single generator seed.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import getpass
import io
import json
import os
import subprocess
import sys
import urllib.error
import urllib.request
import zipfile

drive.mount('/content/drive')
REPOSITORY = 'Zhou-Leheng/adaptive-time-frequency-generation'
request = urllib.request.Request(
    f'https://api.github.com/repos/{REPOSITORY}/zipball/main',
    headers={'Accept': 'application/vnd.github+json'},
)
try:
    response = urllib.request.urlopen(request, timeout=90)
except urllib.error.HTTPError as error:
    if error.code not in (401, 404):
        raise
    try:
        github_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        github_token = getpass.getpass('GitHub token with repository contents read access: ')
    request.add_header('Authorization', f'Bearer {github_token}')
    response = urllib.request.urlopen(request, timeout=90)
    del github_token

with response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
del request, response
root = Path('/content/atfg_code')
root.mkdir(exist_ok=True)
for member in archive.infolist():
    target = (root / member.filename).resolve()
    if not target.is_relative_to(root.resolve()):
        raise ValueError('Unexpected archive path')
archive.extractall(root)
CODE = root / archive.namelist()[0].split('/')[0]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-benchmark.txt')])
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before starting the experiment.'
print('GPU:', torch.cuda.get_device_name(0))
print('Code:', CODE)

In [ ]:
RUN_NAME = 'etth1_seed42'
COMPARISON_NAME = 'benchmark_diffusion_ts'
RESULTS = Path('/content/drive/MyDrive/ATFG') / RUN_NAME
assert (RESULTS / 'settings.json').is_file(), 'Select the folder from the completed fusion experiment.'
source_settings = json.loads((RESULTS / 'settings.json').read_text())
BASELINE_STEPS = source_settings['steps']
EVALUATOR_REPEATS = 3
OUTPUT = RESULTS / COMPARISON_NAME
LOG = Path('/content') / f'atfg_compare_{RUN_NAME}.log'
PID_RECORD = Path('/content') / f'atfg_compare_{RUN_NAME}_pid.json'
print('Saved fusion results:', RESULTS)
print('Baseline optimizer steps:', BASELINE_STEPS)
print('Comparison output:', OUTPUT)


In [ ]:
from filelock import FileLock
with FileLock(str(RESULTS / '.run.lock'), timeout=0):
    pass
command = [
    'nohup', sys.executable, '-u', str(CODE / 'compare.py'),
    '--results', str(RESULTS), '--name', COMPARISON_NAME,
    '--download', '--data', '/content/ETTh1.csv',
    '--baseline-steps', str(BASELINE_STEPS), '--repeats', str(EVALUATOR_REPEATS),
]
with LOG.open('a') as handle:
    process = subprocess.Popen(command, cwd=CODE, stdout=handle, stderr=subprocess.STDOUT,
                               stdin=subprocess.DEVNULL, start_new_session=True)
PID_RECORD.write_text(json.dumps({'pid': process.pid, 'script': str(CODE / 'compare.py')}))
print('Comparison PID:', process.pid)
print('Rerun the next cell to read the log.')


In [ ]:
if LOG.exists():
    print(LOG.read_text(errors='replace')[-8000:])
else:
    print('The log has not been created yet.')

In [ ]:
import pandas as pd
from IPython.display import display
summary = OUTPUT / 'comparison.csv'
if summary.exists():
    frame = pd.read_csv(summary)
    display(frame[['method', 'context_fid', 'discriminative_mean',
                   'predictive_mean', 'correlational_score']])
    print('Complete CSV:', summary)
else:
    print('The baseline or first evaluation is still running. Check the log above.')


Rerun the log cell while the comparison is active. The same launch command resumes baseline checkpoints and reuses completed evaluator stages. To stop at a safe boundary, set STOP = True below and run the cell. Use a new COMPARISON_NAME when changing baseline or evaluation settings.


In [ ]:
STOP = False
if STOP:
    import signal
    record = json.loads(PID_RECORD.read_text())
    command_line = Path(f"/proc/{record['pid']}/cmdline")
    if command_line.exists() and record['script'].encode() in command_line.read_bytes().split(b'\x00'):
        os.kill(record['pid'], signal.SIGTERM)
        print('Stop requested. Check the log for the checkpoint confirmation.')
    else:
        print('The recorded training process has already exited.')